# Dots & Boxes — DQN

Dueling **Double DQN** with n-step returns, trained by self-play on the 5x5 board
(25 boxes, 60 lines). Two runs in parallel, one per GPU:

| run | learning rate | n-step | exploration decay | target sync | updates/burst |
|-----|---------------|--------|-------------------|-------------|---------------|
| **dqn_a** (GPU 0) | 3e-4 | 3 | 4.0M steps | 2000 | 8 |
| **dqn_b** (GPU 1) | 1e-3 | 5 | 1.5M steps | 1000 | 12 |

Both are evaluated against a fixed heuristic bot every 20 minutes so the learning
curves are comparable. Settings: **Accelerator = GPU T4 x2**, then
*Save Version -> Save & Run All (Commit)* so it keeps running with the tab closed.


In [ ]:
%%writefile /kaggle/working/game.py
"""Dots & Boxes as a vectorised RL environment.

Boards are counted in BOXES: N=5 -> 5x5 boxes, 60 lines, 25 boxes.
Everything is batched: one call steps G independent games at once, which is what
keeps a GPU busy during DQN self-play.

Convention used everywhere in this package
------------------------------------------
Rewards and values are always from the point of view of THE PLAYER TO MOVE in
the state they are attached to, expressed as a fraction of the total boxes.
The reward for a move is (boxes that move captured) / B, so the rewards of a
whole game, with the sign flipped every time the turn changes, sum exactly to
the final margin / B.  Completing a box gives you another turn; that is why the
turn does not simply alternate and why `same_mover` is returned by `step`.
"""
import numpy as np
from numba import njit

__all__ = ["Geometry", "Batch", "C_IN", "heuristic_moves"]

C_IN = 20   # see Batch.encode for the plane list


class Geometry:
    """Line/box index maps for an N x N board, plus the 8 board symmetries."""

    def __init__(self, N):
        assert 2 <= N <= 7
        self.N = N
        self.B = B = N * N
        self.L = L = 2 * N * (N + 1)
        self.S = S = 2 * N + 1
        self.NH = N * (N + 1)

        line_cell = np.zeros((L, 2), np.int32)
        for r in range(N + 1):
            for c in range(N):
                line_cell[self.h(r, c)] = (2 * r, 2 * c + 1)
        for r in range(N):
            for c in range(N + 1):
                line_cell[self.v(r, c)] = (2 * r + 1, 2 * c)

        box_cell = np.zeros((B, 2), np.int32)
        box_lines = np.zeros((B, 4), np.int32)
        for r in range(N):
            for c in range(N):
                b = r * N + c
                box_cell[b] = (2 * r + 1, 2 * c + 1)
                box_lines[b] = (self.h(r, c), self.h(r + 1, c),
                                self.v(r, c), self.v(r, c + 1))

        line_box = np.full((L, 2), -1, np.int32)
        for b in range(B):
            for l in box_lines[b]:
                if line_box[l, 0] < 0:
                    line_box[l, 0] = b
                else:
                    line_box[l, 1] = b

        cell_line = np.full((S, S), -1, np.int32)
        cell_box = np.full((S, S), -1, np.int32)
        for l in range(L):
            cell_line[line_cell[l, 0], line_cell[l, 1]] = l
        for b in range(B):
            cell_box[box_cell[b, 0], box_cell[b, 1]] = b

        line_perm = np.zeros((8, L), np.int32)
        box_perm = np.zeros((8, B), np.int32)
        for t in range(8):
            for l in range(L):
                r, c = self._tf(t, line_cell[l, 0], line_cell[l, 1])
                line_perm[t, l] = cell_line[r, c]
            for b in range(B):
                r, c = self._tf(t, box_cell[b, 0], box_cell[b, 1])
                box_perm[t, b] = cell_box[r, c]
        assert (line_perm >= 0).all() and (box_perm >= 0).all()

        self.line_cell, self.box_cell = line_cell, box_cell
        self.box_lines, self.line_box = box_lines, line_box
        self.line_perm, self.box_perm = line_perm, box_perm
        self.line_rc = (line_cell[:, 0], line_cell[:, 1])
        self.box_rc = (box_cell[:, 0], box_cell[:, 1])

    def h(self, r, c):
        return r * self.N + c

    def v(self, r, c):
        return self.NH + r * (self.N + 1) + c

    def _tf(self, t, r, c):
        m = self.S - 1
        return [(r, c), (c, m - r), (m - r, m - c), (m - c, r),
                (r, m - c), (m - r, c), (c, r), (m - c, m - r)][t]

    def describe(self, l):
        if l < self.NH:
            return "h %d %d" % divmod(l, self.N)
        r, c = divmod(l - self.NH, self.N + 1)
        return "v %d %d" % (r, c)


# ---------------------------------------------------------------- numba core

@njit(cache=True)
def _apply(drawn, sides, owner, score, mover, done, actions, line_box, B):
    """Apply one action per game. Returns (captures, same_mover)."""
    G = drawn.shape[0]
    caps = np.zeros(G, np.int64)
    same = np.zeros(G, np.uint8)
    for g in range(G):
        if done[g]:
            continue
        m = actions[g]
        drawn[g, m] = 1
        k = 0
        for j in range(2):
            b = line_box[m, j]
            if b >= 0:
                sides[g, b] += 1
                if sides[g, b] == 4:
                    owner[g, b] = mover[g]
                    k += 1
        caps[g] = k
        if k > 0:
            score[g, mover[g] - 1] += k
            same[g] = 1
        else:
            mover[g] = 3 - mover[g]
        nleft = 0
        for b in range(B):
            if sides[g, b] < 4:
                nleft = 1
                break
        if nleft == 0:
            done[g] = True
    return caps, same


@njit(cache=True)
def _components(drawn, sides, line_box, box_lines, B, comp, size, formed, ground, stack):
    """Connected groups of open boxes joined by undrawn lines, per game.

    comp[g,b]   component id of box b (-1 if the box is finished)
    size[g,c]   number of boxes in component c
    formed[g,c] 1 if every box in it already has exactly 2 sides (a real chain/loop)
    ground[g,c] number of undrawn lines leaving the component to the board edge
    Returns the component count per game.
    """
    G = drawn.shape[0]
    ncomp = np.zeros(G, np.int64)
    for g in range(G):
        for b in range(B):
            comp[g, b] = -1
        nc = 0
        for s in range(B):
            if sides[g, s] >= 4 or comp[g, s] >= 0:
                continue
            comp[g, s] = nc
            size[g, nc] = 0
            formed[g, nc] = 1
            ground[g, nc] = 0
            top = 0
            stack[g, top] = s
            top += 1
            while top > 0:
                top -= 1
                b = stack[g, top]
                size[g, nc] += 1
                if sides[g, b] != 2:
                    formed[g, nc] = 0
                for j in range(4):
                    l = box_lines[b, j]
                    if drawn[g, l] != 0:
                        continue
                    o = line_box[l, 0]
                    if o == b:
                        o = line_box[l, 1]
                    if o < 0:
                        ground[g, nc] += 1
                    elif comp[g, o] < 0:
                        comp[g, o] = nc
                        stack[g, top] = o
                        top += 1
            nc += 1
        ncomp[g] = nc
    return ncomp


@njit(cache=True)
def _heuristic(drawn, sides, line_box, box_lines, L, B, done, out):
    """A decent casual opponent: take free boxes, then play safe, else give away
    as few boxes as possible.  Fixed forever so eval numbers stay comparable."""
    G = drawn.shape[0]
    for g in range(G):
        if done[g]:
            out[g] = -1
            continue
        pick = -1
        # a capture that does not open another box
        for b in range(B):
            if sides[g, b] == 3:
                l = -1
                for j in range(4):
                    if drawn[g, box_lines[b, j]] == 0:
                        l = box_lines[b, j]
                        break
                o = line_box[l, 0]
                if o == b:
                    o = line_box[l, 1]
                if o < 0 or sides[g, o] != 2:
                    pick = l
                    break
        if pick < 0:
            for b in range(B):
                if sides[g, b] == 3:
                    for j in range(4):
                        if drawn[g, box_lines[b, j]] == 0:
                            pick = box_lines[b, j]
                            break
                    break
        if pick < 0:                       # a safe line
            n = 0
            for l in range(L):
                if drawn[g, l] == 0:
                    ok = True
                    for j in range(2):
                        b = line_box[l, j]
                        if b >= 0 and sides[g, b] >= 2:
                            ok = False
                    if ok:
                        n += 1
            if n > 0:
                pickn = np.random.randint(n)
                n = 0
                for l in range(L):
                    if drawn[g, l] == 0:
                        ok = True
                        for j in range(2):
                            b = line_box[l, j]
                            if b >= 0 and sides[g, b] >= 2:
                                ok = False
                        if ok:
                            if n == pickn:
                                pick = l
                                break
                            n += 1
        if pick < 0:                       # open the smallest chain
            best = 10000
            for l in range(L):
                if drawn[g, l] == 0:
                    c = 0
                    for j in range(2):
                        b = line_box[l, j]
                        if b >= 0 and sides[g, b] == 2:
                            c += 1
                    if c < best:
                        best = c
                        pick = l
        out[g] = pick


# ------------------------------------------------------------------- wrapper

class Batch:
    """G games played in lockstep."""

    def __init__(self, geo, G, rng=None):
        self.geo, self.G = geo, G
        self.rng = rng or np.random.default_rng()
        L, B = geo.L, geo.B
        self.drawn = np.zeros((G, L), np.uint8)
        self.sides = np.zeros((G, B), np.int8)
        self.owner = np.zeros((G, B), np.int8)
        self.score = np.zeros((G, 2), np.int32)
        self.mover = np.ones(G, np.int8)
        self.done = np.zeros(G, np.bool_)
        self._comp = np.zeros((G, B), np.int64)
        self._size = np.zeros((G, B + 1), np.int64)
        self._formed = np.zeros((G, B + 1), np.int64)
        self._ground = np.zeros((G, B + 1), np.int64)
        self._stack = np.zeros((G, B + 1), np.int64)
        self._hbuf = np.zeros(G, np.int64)

    def reset(self, idx=None):
        idx = slice(None) if idx is None else idx
        self.drawn[idx] = 0
        self.sides[idx] = 0
        self.owner[idx] = 0
        self.score[idx] = 0
        self.mover[idx] = 1
        self.done[idx] = False

    def clone_state(self):
        return (self.drawn.copy(), self.owner.copy(), self.score.copy(),
                self.mover.copy(), self.done.copy())

    def legal(self):
        return self.drawn == 0

    def step(self, actions):
        caps, same = _apply(self.drawn, self.sides, self.owner, self.score,
                            self.mover, self.done, np.ascontiguousarray(actions, np.int64),
                            self.geo.line_box, self.geo.B)
        return caps / self.geo.B, same.astype(bool)

    def margin(self):
        """Final margin from player 1's point of view."""
        return self.score[:, 0] - self.score[:, 1]

    def heuristic(self):
        _heuristic(self.drawn, self.sides, self.geo.line_box, self.geo.box_lines,
                   self.geo.L, self.geo.B, self.done, self._hbuf)
        return self._hbuf.copy()

    # ---- observation ----
    def encode(self, out=None):
        if not hasattr(self, "_enc"):
            self._enc = Encoder(self.geo, self.G)
        return self._enc(self.drawn, self.sides, self.owner, self.score, self.mover, out)


class Encoder:
    """Turns raw board arrays into network input. Reused for self-play and for
    replay batches, so training and acting always see identical features."""

    def __init__(self, geo, capacity):
        self.geo = geo
        B = geo.B
        self.cap = capacity
        self._comp = np.zeros((capacity, B), np.int64)
        self._size = np.zeros((capacity, B + 1), np.int64)
        self._formed = np.zeros((capacity, B + 1), np.int64)
        self._ground = np.zeros((capacity, B + 1), np.int64)
        self._stack = np.zeros((capacity, B + 1), np.int64)

    def __call__(self, drawn, sides, owner, score, mover, out=None):
        """(G, C_IN, S, S) float32.

        0  on board            1  line drawn          2  line undrawn
        3  line completes a box                       4  line gives a box away
        5..8  box has 0/1/2/3 sides                   9  box owned by mover
        10 box owned by opponent                     11 dots
        12 score difference / B (mover's view)       13 lines left / L
        14 safe lines left / L                       15 parity of lines left
        -- chain structure, the part a plain conv net cannot work out itself --
        16 size of this box's component / B          17 component is fully formed
        18 long chains on the board / B              19 PARITY of that count
        """
        geo = self.geo
        S, L, B = geo.S, geo.L, geo.B
        G = drawn.shape[0]
        assert G <= self.cap, "encoder scratch too small"
        X = np.zeros((G, C_IN, S, S), np.float32) if out is None else out[:G]
        X[:] = 0.0
        lr, lc = geo.line_rc
        br, bc = geo.box_rc

        X[:, 0] = 1.0
        X[:, 11, ::2, ::2] = 1.0
        X[:, 1][:, lr, lc] = drawn
        X[:, 2][:, lr, lc] = 1 - drawn

        s_of_line = sides[:, geo.line_box]
        valid = geo.line_box[None, :, :] >= 0
        cap = ((s_of_line == 3) & valid).any(2) & (drawn == 0)
        giv = ((s_of_line == 2) & valid).any(2) & (drawn == 0)
        X[:, 3][:, lr, lc] = cap
        X[:, 4][:, lr, lc] = giv
        safe = (drawn == 0) & ~cap & ~giv

        for k in range(4):
            X[:, 5 + k][:, br, bc] = (sides == k)
        mine = owner == mover[:, None]
        X[:, 9][:, br, bc] = (sides == 4) & mine
        X[:, 10][:, br, bc] = (sides == 4) & ~mine & (owner != 0)

        diff = (score[:, 0].astype(np.float32) - score[:, 1])
        diff = np.where(mover == 2, -diff, diff) / B
        rem = (drawn == 0).sum(1).astype(np.float32)
        X[:, 12] = diff[:, None, None]
        X[:, 13] = (rem / L)[:, None, None]
        X[:, 14] = (safe.sum(1) / L).astype(np.float32)[:, None, None]
        X[:, 15] = (rem % 2)[:, None, None]

        nc = _components(np.ascontiguousarray(drawn), np.ascontiguousarray(sides),
                         geo.line_box, geo.box_lines, B, self._comp[:G], self._size[:G],
                         self._formed[:G], self._ground[:G], self._stack[:G])
        open_ = sides < 4
        gi = np.arange(G)[:, None]
        cid = np.where(open_, self._comp[:G], 0)
        X[:, 16][:, br, bc] = np.where(open_, self._size[:G][gi, cid] / B, 0.0)
        X[:, 17][:, br, bc] = np.where(open_, self._formed[:G][gi, cid], 0)

        nlong = np.zeros(G, np.float32)
        for g in range(G):
            k = nc[g]
            if k:
                sz = self._size[g, :k]
                fm = self._formed[g, :k]
                gr = self._ground[g, :k]
                nlong[g] = int(((fm == 1) & (gr == 2) & (sz >= 3)).sum())
        X[:, 18] = (nlong / B)[:, None, None]
        X[:, 19] = (nlong % 2)[:, None, None]
        return X


def sides_from_drawn(geo, drawn):
    """Recover per-box drawn-side counts from a batch of line vectors."""
    return drawn[:, geo.box_lines].sum(2).astype(np.int8)


In [ ]:
%%writefile /kaggle/working/model.py
"""Dueling Double-DQN network for Dots & Boxes.

Q(s, a) is the expected REMAINING margin for the player to move in s, divided by
the number of boxes, so it lives in [-1, 1] and is directly comparable across
board sizes.  Dueling split: Q = V(s) + A(s,a) - mean_legal A(s, .)
"""
import torch
import torch.nn as nn
import torch.nn.functional as F

from game import C_IN


class GlobalPoolBias(nn.Module):
    """Lets every cell see board-wide facts (parity, score) that a 3x3 kernel
    can never reach on its own."""

    def __init__(self, ch):
        super().__init__()
        self.fc = nn.Linear(2 * ch, ch)

    def forward(self, x):
        g = torch.cat([x.mean(dim=(2, 3)), x.amax(dim=(2, 3))], dim=1)
        return self.fc(g)[:, :, None, None]


class ResBlock(nn.Module):
    def __init__(self, ch, gpool):
        super().__init__()
        self.c1 = nn.Conv2d(ch, ch, 3, padding=1, bias=False)
        self.b1 = nn.BatchNorm2d(ch)
        self.c2 = nn.Conv2d(ch, ch, 3, padding=1, bias=False)
        self.b2 = nn.BatchNorm2d(ch)
        self.gp = GlobalPoolBias(ch) if gpool else None

    def forward(self, x):
        y = F.relu(self.b1(self.c1(x)))
        if self.gp is not None:
            y = y + self.gp(y)
        return F.relu(x + self.b2(self.c2(y)))


class DuelingQNet(nn.Module):
    def __init__(self, channels=64, blocks=6):
        super().__init__()
        self.channels, self.blocks = channels, blocks
        self.stem = nn.Sequential(nn.Conv2d(C_IN, channels, 3, padding=1, bias=False),
                                  nn.BatchNorm2d(channels), nn.ReLU())
        self.trunk = nn.Sequential(*[ResBlock(channels, gpool=(i % 3 == 2))
                                     for i in range(blocks)])
        self.adv = nn.Sequential(nn.Conv2d(channels, 32, 1, bias=False),
                                 nn.BatchNorm2d(32), nn.ReLU(), nn.Conv2d(32, 1, 1))
        self.val_conv = nn.Sequential(nn.Conv2d(channels, 32, 1, bias=False),
                                      nn.BatchNorm2d(32), nn.ReLU())
        self.val_fc = nn.Sequential(nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 1))

    def forward(self, x, line_flat):
        h = self.trunk(self.stem(x))
        n = x.shape[0]
        a = self.adv(h).reshape(n, -1).index_select(1, line_flat)
        vh = self.val_conv(h)
        vg = torch.cat([vh.mean(dim=(2, 3)), vh.amax(dim=(2, 3))], dim=1)
        v = torch.tanh(self.val_fc(vg)).squeeze(1)
        return a, v


NEG = -1e9


def q_from(adv, v, legal):
    """Combine the dueling heads, with illegal moves removed from the mean and
    pushed to -inf so they can never be selected."""
    legal_f = legal.float()
    n_legal = legal_f.sum(1).clamp(min=1.0)
    adv_z = adv * legal_f
    mean_a = adv_z.sum(1) / n_legal
    q = v[:, None] + adv_z - mean_a[:, None]
    return q.masked_fill(~legal, NEG)


In [ ]:
%%writefile /kaggle/working/buffer.py
"""N-step transition store for two-player DQN.

The turn does not alternate reliably in Dots & Boxes -- completing a box gives
you another move -- so a transition cannot assume the next state belongs to the
opponent.  Every entry therefore carries `sign`: +1 if the bootstrap state is
the SAME player's turn as the state the transition started from, -1 if it is the
opponent's.  The n-step target is

    y = r_acc + sign * gamma^k * max_a Q(s_k, a)        (0 if the game ended)

where r_acc already has the per-step signs folded in.
"""
import numpy as np


class Replay:
    def __init__(self, geo, capacity):
        L, B = geo.L, geo.B
        self.cap = int(capacity)
        self.drawn = np.zeros((self.cap, L), np.uint8)
        self.owner = np.zeros((self.cap, B), np.int8)
        self.score = np.zeros((self.cap, 2), np.int16)
        self.mover = np.zeros(self.cap, np.int8)
        self.action = np.zeros(self.cap, np.int16)
        self.r_acc = np.zeros(self.cap, np.float32)
        self.sign = np.zeros(self.cap, np.int8)
        self.k = np.zeros(self.cap, np.int8)
        self.done = np.zeros(self.cap, np.uint8)
        self.n_drawn = np.zeros((self.cap, L), np.uint8)
        self.n_owner = np.zeros((self.cap, B), np.int8)
        self.n_score = np.zeros((self.cap, 2), np.int16)
        self.n_mover = np.zeros(self.cap, np.int8)
        self.n = self.ptr = self.total = 0

    FIELDS = ("drawn", "owner", "score", "mover", "action", "r_acc", "sign",
              "k", "done", "n_drawn", "n_owner", "n_score", "n_mover")

    def add_many(self, **kw):
        m = len(kw["action"])
        if m == 0:
            return
        idx = (self.ptr + np.arange(m)) % self.cap
        for f in self.FIELDS:
            getattr(self, f)[idx] = kw[f]
        self.ptr = int((self.ptr + m) % self.cap)
        self.n = min(self.cap, self.n + m)
        self.total += m

    def sample(self, batch, rng):
        i = rng.integers(0, self.n, size=batch)
        return {f: getattr(self, f)[i] for f in self.FIELDS}

    def save(self, path):
        tmp = path + ".tmp.npz"
        np.savez_compressed(tmp, n=self.n, ptr=self.ptr, total=self.total,
                            **{f: getattr(self, f)[:self.n] for f in self.FIELDS})
        import os
        os.replace(tmp, path)

    def load(self, path):
        d = np.load(path)
        n = min(int(d["n"]), self.cap)
        for f in self.FIELDS:
            getattr(self, f)[:n] = d[f][-n:] if len(d[f]) > n else d[f][:n]
        self.n = n
        self.ptr = n % self.cap
        self.total = int(d["total"])


class NStepTracker:
    """Per-game sliding window that turns raw 1-step transitions into n-step ones."""

    def __init__(self, G, n, gamma=1.0):
        self.G, self.n, self.gamma = G, int(n), float(gamma)
        self.buf = [[] for _ in range(G)]

    def reset_game(self, g):
        self.buf[g] = []

    def push(self, g, state, action, reward, same_mover):
        """state = (drawn, owner, score, mover) copies for THIS decision."""
        self.buf[g].append([state, action, reward, same_mover])

    def pop_ready(self, g, next_state, terminal):
        """Emit every transition whose n-step window is complete.
        On a terminal step everything still buffered is emitted."""
        out = []
        b = self.buf[g]
        limit = len(b) if terminal else len(b) - self.n
        for t in range(max(0, limit)):
            st, a, _, _ = b[t]
            sign = 1
            r_acc = 0.0
            g_pow = 1.0
            steps = 0
            for i in range(t, len(b)):
                r_acc += sign * g_pow * b[i][2]
                g_pow *= self.gamma
                steps += 1
                if not b[i][3]:          # no capture -> the turn changed
                    sign = -sign
                if steps == self.n:
                    break
            j = t + steps
            if j < len(b):               # bootstrap from a state we still hold
                ns, done = b[j][0], 0
            else:                        # ran off the end of the trajectory
                ns, done = next_state, (1 if terminal else 0)
            out.append(dict(drawn=st[0], owner=st[1], score=st[2], mover=st[3],
                            action=a, r_acc=r_acc, sign=sign, k=steps, done=done,
                            n_drawn=ns[0], n_owner=ns[1], n_score=ns[2], n_mover=ns[3]))
        if terminal:
            self.buf[g] = []
        else:
            del b[:max(0, limit)]
        return out


In [ ]:
%%writefile /kaggle/working/train_dqn.py
#!/usr/bin/env python3
"""Dueling Double-DQN with n-step returns for Dots & Boxes.

  python train_dqn.py --board 5 --out runs/dqn5 --hours 8.5
  python train_dqn.py --eval-only --resume-from runs/dqn5

Design notes
------------
* Self-play: both sides act eps-greedily from the SAME online network, so the
  opponent improves with the agent.  Transitions are always stored from the
  point of view of the player to move.
* Two-player bootstrap: completing a box means you move again, so the target
  carries an explicit sign instead of assuming the turn alternated (buffer.py).
* Reward shaping: r = boxes captured by this move / B.  Signed and summed over a
  game this telescopes to the exact final margin / B, so the shaping is exact,
  not a heuristic -- it just moves the signal earlier, which is the single
  biggest problem a value-based method has on this game.
* Double DQN: the online net picks the bootstrap action, the target net scores
  it, which removes most of the max-operator optimism bias.
"""
import os, sys, time, json, math, signal, argparse
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import game as G
from game import Geometry, Batch, Encoder, sides_from_drawn, C_IN
from model import DuelingQNet, q_from
from buffer import Replay, NStepTracker


def log_to(path):
    def log(msg):
        line = time.strftime("%H:%M:%S ") + msg
        print(line, flush=True)
        try:
            with open(path, "a") as f:
                f.write(line + "\n")
        except OSError:
            pass
    return log


def atomic_save(obj, path):
    torch.save(obj, path + ".tmp")
    os.replace(path + ".tmp", path)


class Runner:
    def __init__(self, args):
        self.args = args
        self.geo = geo = Geometry(args.board)
        self.dev = torch.device(args.device if args.device != "auto" else
                                ("cuda" if torch.cuda.is_available() else "cpu"))
        self.rng = np.random.default_rng(args.seed)
        torch.manual_seed(args.seed)

        self.net = DuelingQNet(args.channels, args.blocks).to(self.dev)
        self.tgt = DuelingQNet(args.channels, args.blocks).to(self.dev)
        self.tgt.load_state_dict(self.net.state_dict())
        self.tgt.eval()
        self.opt = torch.optim.AdamW(self.net.parameters(), lr=args.lr, weight_decay=1e-4)
        self.scaler = torch.amp.GradScaler("cuda", enabled=self.dev.type == "cuda")
        self.line_flat = torch.as_tensor(
            (geo.line_cell[:, 0] * geo.S + geo.line_cell[:, 1]).astype(np.int64), device=self.dev)

        self.replay = Replay(geo, args.buffer)
        self.batch = Batch(geo, args.games, self.rng)
        self.tracker = NStepTracker(args.games, args.nstep, args.gamma)
        self.enc_play = Encoder(geo, args.games)
        self.enc_train = Encoder(geo, args.batch)
        self.state = dict(steps=0, updates=0, games=0, hours=0.0, best_vs_bot=-1.0)

    # ------------------------------------------------------------ act
    @torch.inference_mode()
    def q_of(self, drawn, sides, owner, score, mover, enc, net=None):
        X = enc(drawn, sides, owner, score, mover)
        x = torch.from_numpy(X).to(self.dev)
        legal = torch.from_numpy(drawn == 0).to(self.dev)
        with torch.autocast(device_type=self.dev.type, dtype=torch.float16,
                            enabled=self.dev.type == "cuda"):
            a, v = (net or self.net)(x, self.line_flat)
        return q_from(a.float(), v.float(), legal)

    def eps_now(self):
        a = self.args
        f = min(1.0, self.state["steps"] / max(1, a.eps_decay_steps))
        return a.eps_start + (a.eps_end - a.eps_start) * f

    def collect(self, n_steps):
        """Run n_steps of self-play across all parallel games."""
        b, geo, a = self.batch, self.geo, self.args
        self.net.eval()
        eps = self.eps_now()
        pending = {f: [] for f in Replay.FIELDS}
        for _ in range(n_steps):
            live = ~b.done
            if not live.any():
                b.reset(); [self.tracker.reset_game(g) for g in range(a.games)]
                continue
            q = self.q_of(b.drawn, b.sides, b.owner, b.score, b.mover, self.enc_play)
            greedy = q.argmax(1).cpu().numpy()
            legal = b.legal()
            explore = self.rng.random(a.games) < eps
            act = greedy.copy()
            for g in np.flatnonzero(explore & live):
                cand = np.flatnonzero(legal[g])
                act[g] = cand[self.rng.integers(len(cand))]
            snap = (b.drawn.copy(), b.owner.copy(), b.score.astype(np.int16).copy(), b.mover.copy())
            for g in np.flatnonzero(live):
                self.tracker.push(g, (snap[0][g], snap[1][g], snap[2][g], snap[3][g]),
                                  int(act[g]), 0.0, True)
            rew, same = b.step(act)
            for g in np.flatnonzero(live):
                self.tracker.buf[g][-1][2] = float(rew[g])
                self.tracker.buf[g][-1][3] = bool(same[g])
            nxt = (b.drawn, b.owner, b.score.astype(np.int16), b.mover)
            for g in np.flatnonzero(live):
                term = bool(b.done[g])
                # copy: these rows are about to be reset for finished games
                ns = (nxt[0][g].copy(), nxt[1][g].copy(), nxt[2][g].copy(), int(nxt[3][g]))
                for tr in self.tracker.pop_ready(g, ns, term):
                    for f in Replay.FIELDS:
                        pending[f].append(tr[f])
                if term:
                    self.state["games"] += 1
            self.state["steps"] += int(live.sum())
            fin = np.flatnonzero(b.done)
            if len(fin):
                b.reset(fin)
                for g in fin:
                    self.tracker.reset_game(int(g))
        if pending["action"]:
            self.replay.add_many(**{f: np.array(pending[f]) for f in Replay.FIELDS})

    # ------------------------------------------------------------ learn
    def update(self):
        a, geo = self.args, self.geo
        s = self.replay.sample(a.batch, self.rng)
        sides = sides_from_drawn(geo, s["drawn"])
        n_sides = sides_from_drawn(geo, s["n_drawn"])
        X = self.enc_train(s["drawn"], sides, s["owner"], s["score"], s["mover"])
        NX = self.enc_train(s["n_drawn"], n_sides, s["n_owner"], s["n_score"], s["n_mover"])
        x = torch.from_numpy(X.copy()).to(self.dev)
        nx = torch.from_numpy(NX.copy()).to(self.dev)
        legal = torch.from_numpy(s["drawn"] == 0).to(self.dev)
        n_legal = torch.from_numpy(s["n_drawn"] == 0).to(self.dev)
        act = torch.from_numpy(s["action"].astype(np.int64)).to(self.dev)
        r = torch.from_numpy(s["r_acc"]).to(self.dev)
        sign = torch.from_numpy(s["sign"].astype(np.float32)).to(self.dev)
        done = torch.from_numpy(s["done"].astype(np.float32)).to(self.dev)
        disc = torch.from_numpy((a.gamma ** s["k"]).astype(np.float32)).to(self.dev)

        with torch.no_grad():
            with torch.autocast(device_type=self.dev.type, dtype=torch.float16,
                                enabled=self.dev.type == "cuda"):
                na_on, nv_on = self.net(nx, self.line_flat)
                na_tg, nv_tg = self.tgt(nx, self.line_flat)
            best = q_from(na_on.float(), nv_on.float(), n_legal).argmax(1, keepdim=True)
            q_next = q_from(na_tg.float(), nv_tg.float(), n_legal).gather(1, best).squeeze(1)
            q_next = torch.where(n_legal.any(1), q_next, torch.zeros_like(q_next))
            y = r + (1.0 - done) * sign * disc * q_next
            y = y.clamp(-1.0, 1.0)

        self.net.train()
        with torch.autocast(device_type=self.dev.type, dtype=torch.float16,
                            enabled=self.dev.type == "cuda"):
            adv, v = self.net(x, self.line_flat)
        q = q_from(adv.float(), v.float(), legal).gather(1, act[:, None]).squeeze(1)
        loss = F.smooth_l1_loss(q, y, beta=0.1)
        self.opt.zero_grad(set_to_none=True)
        self.scaler.scale(loss).backward()
        self.scaler.unscale_(self.opt)
        nn.utils.clip_grad_norm_(self.net.parameters(), 5.0)
        self.scaler.step(self.opt)
        self.scaler.update()
        self.state["updates"] += 1
        if self.state["updates"] % a.target_sync == 0:
            self.tgt.load_state_dict(self.net.state_dict())
        return float(loss.detach()), float(y.mean()), float(q.detach().mean())

    # ------------------------------------------------------------ eval
    @torch.inference_mode()
    def eval_vs_bot(self, n_games, seed=0):
        """Greedy DQN against the fixed heuristic bot. No search on either side."""
        geo, a = self.geo, self.args
        n_games += n_games % 2
        rng = np.random.default_rng(seed)
        b = Batch(geo, n_games, rng)
        enc = Encoder(geo, n_games)
        side = np.where(np.arange(n_games) % 2 == 0, 1, 2).astype(np.int8)
        self.net.eval()
        while not b.done.all():
            ours = (b.mover == side) & ~b.done
            theirs = (~ours) & ~b.done
            act = np.zeros(n_games, np.int64)
            if theirs.any():
                act[theirs] = b.heuristic()[theirs]
            if ours.any():
                q = self.q_of(b.drawn, b.sides, b.owner, b.score, b.mover, enc)
                act[ours] = q.argmax(1).cpu().numpy()[ours]
            act = np.where(b.done, 0, act)
            b.step(act)
        mg = np.where(side == 1, b.margin(), -b.margin())
        return dict(games=n_games, wins=int((mg > 0).sum()), draws=int((mg == 0).sum()),
                    losses=int((mg < 0).sum()), score=float(((mg > 0).sum() + 0.5 * (mg == 0).sum()) / n_games),
                    avg_margin=float(mg.mean()))

    # ------------------------------------------------------------ io
    def save(self, out, tag):
        atomic_save(dict(net=self.net.state_dict(), tgt=self.tgt.state_dict(),
                         opt=self.opt.state_dict(), scaler=self.scaler.state_dict(),
                         state=dict(self.state), board=self.geo.N,
                         cfg=dict(channels=self.args.channels, blocks=self.args.blocks)),
                    os.path.join(out, "latest.pt"))

    def load(self, path, log):
        ck = torch.load(path, map_location=self.dev, weights_only=False)
        self.net.load_state_dict(ck["net"])
        self.tgt.load_state_dict(ck.get("tgt", ck["net"]))
        if "opt" in ck:
            self.opt.load_state_dict(ck["opt"])
        if "scaler" in ck:
            self.scaler.load_state_dict(ck["scaler"])
        self.state.update(ck.get("state", {}))
        log(f"resumed {path}: {self.state['steps']:,} env steps, "
            f"{self.state['updates']:,} updates, {self.state['hours']:.2f} h")


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--board", type=int, default=5)
    p.add_argument("--out", default="runs/dqn5")
    p.add_argument("--hours", type=float, default=8.5)
    p.add_argument("--games", type=int, default=256, help="parallel self-play games")
    p.add_argument("--channels", type=int, default=64)
    p.add_argument("--blocks", type=int, default=6)
    p.add_argument("--lr", type=float, default=3e-4)
    p.add_argument("--batch", type=int, default=512)
    p.add_argument("--buffer", type=int, default=400_000)
    p.add_argument("--nstep", type=int, default=3)
    p.add_argument("--gamma", type=float, default=1.0)
    p.add_argument("--eps-start", type=float, default=1.0)
    p.add_argument("--eps-end", type=float, default=0.05)
    p.add_argument("--eps-decay-steps", type=int, default=3_000_000)
    p.add_argument("--target-sync", type=int, default=2000)
    p.add_argument("--collect-steps", type=int, default=8, help="env steps between update bursts")
    p.add_argument("--updates", type=int, default=8, help="gradient steps per burst")
    p.add_argument("--min-buffer", type=int, default=50_000)
    p.add_argument("--eval-minutes", type=float, default=20.0)
    p.add_argument("--eval-games", type=int, default=200)
    p.add_argument("--save-minutes", type=float, default=15.0)
    p.add_argument("--resume-from", default=None)
    p.add_argument("--device", default="auto")
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--eval-only", action="store_true")
    a = p.parse_args()

    os.makedirs(a.out, exist_ok=True)
    log = log_to(os.path.join(a.out, "train_log.txt"))
    r = Runner(a)
    latest = os.path.join(a.out, "latest.pt")
    src = a.resume_from and os.path.join(a.resume_from, "latest.pt")
    if os.path.exists(latest):
        r.load(latest, log)
    elif src and os.path.exists(src):
        r.load(src, log)
    rep_path = os.path.join(a.out, "replay.npz")
    if os.path.exists(rep_path):
        r.replay.load(rep_path)
        log(f"replay loaded: {r.replay.n:,}")

    if a.eval_only:
        res = r.eval_vs_bot(a.eval_games, seed=7)
        log(f"EVAL vs heuristic bot: W{res['wins']} D{res['draws']} L{res['losses']} "
            f"score {res['score']:.3f} margin {res['avg_margin']:+.2f}")
        return

    log(f"DQN {a.board}x{a.board} | device {r.dev} | dueling double DQN, n-step {a.nstep}, "
        f"gamma {a.gamma} | {a.games} parallel games | net {a.channels}x{a.blocks} "
        f"({sum(x.numel() for x in r.net.parameters()):,} params) | budget {a.hours:.2f} h")

    t0 = time.time()
    hours_before = r.state["hours"]
    deadline = t0 + a.hours * 3600
    stop = {"v": False}
    try:
        signal.signal(signal.SIGTERM, lambda *_: stop.update(v=True))
    except Exception:
        pass
    mpath = os.path.join(a.out, "metrics.csv")
    if not os.path.exists(mpath):
        with open(mpath, "w") as f:
            f.write("hours,env_steps,updates,games,eps,loss,q_mean,target_mean,buffer,vs_bot,vs_bot_margin\n")

    last_save = last_eval = last_log = time.time()
    loss = qm = ym = float("nan")
    best = r.state.get("best_vs_bot", -1.0)
    vs_bot = vs_margin = ""
    while time.time() < deadline and not stop["v"]:
        if os.path.exists(os.path.join(a.out, "STOP")):
            log("stop requested"); break
        r.collect(a.collect_steps)
        if r.replay.n >= a.min_buffer:
            for _ in range(a.updates):
                loss, ym, qm = r.update()
        now = time.time()
        el = hours_before + (now - t0) / 3600
        if now - last_eval > a.eval_minutes * 60 and r.replay.n >= a.min_buffer:
            last_eval = now
            res = r.eval_vs_bot(a.eval_games, seed=r.state["updates"])
            vs_bot, vs_margin = f"{res['score']:.3f}", f"{res['avg_margin']:.2f}"
            tag = ""
            if res["score"] > best:
                best = res["score"]
                r.state["best_vs_bot"] = best
                atomic_save(dict(net=r.net.state_dict(), state=dict(r.state), board=a.board,
                                 cfg=dict(channels=a.channels, blocks=a.blocks), vs_bot=res),
                            os.path.join(a.out, "best.pt"))
                tag = "  -> NEW BEST"
            log(f"EVAL vs heuristic bot: W{res['wins']} D{res['draws']} L{res['losses']} "
                f"score {res['score']:.3f} margin {res['avg_margin']:+.2f}{tag}")
        if now - last_log > 60:
            last_log = now
            log(f"{el:5.2f}h | env steps {r.state['steps']:,} | updates {r.state['updates']:,} | "
                f"games {r.state['games']:,} | eps {r.eps_now():.3f} | loss {loss:.4f} | "
                f"Q {qm:+.3f} target {ym:+.3f} | buffer {r.replay.n:,}")
            with open(mpath, "a") as f:
                f.write(f"{el:.3f},{r.state['steps']},{r.state['updates']},{r.state['games']},"
                        f"{r.eps_now():.4f},{loss:.5f},{qm:.4f},{ym:.4f},{r.replay.n},"
                        f"{vs_bot},{vs_margin}\n")
            vs_bot = vs_margin = ""
            try:
                with open(os.path.join(a.out, "status.json"), "w") as f:
                    json.dump(dict(board=a.board, hours=round(el, 3), env_steps=r.state["steps"],
                                   updates=r.state["updates"], games=r.state["games"],
                                   eps=round(r.eps_now(), 4), loss=loss, best_vs_bot=best,
                                   buffer=r.replay.n, time=time.strftime("%Y-%m-%d %H:%M:%S")), f)
            except OSError:
                pass
        if now - last_save > a.save_minutes * 60:
            last_save = now
            r.state["hours"] = el
            r.save(a.out, "periodic")
            r.replay.save(rep_path)
            log("checkpoint saved")
    r.state["hours"] = hours_before + (time.time() - t0) / 3600
    r.save(a.out, "final")
    r.replay.save(rep_path)
    res = r.eval_vs_bot(a.eval_games, seed=999)
    log(f"FINAL vs heuristic bot: W{res['wins']} D{res['draws']} L{res['losses']} "
        f"score {res['score']:.3f} margin {res['avg_margin']:+.2f}")
    log(f"done: {r.state['hours']:.2f} h, {r.state['games']:,} games, {r.state['updates']:,} updates")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/selftest.py
"""Fast correctness checks for the DQN game core. Runs before any training."""
import sys, os, numpy as np, torch
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import game as G
from game import Geometry, Batch, Encoder, sides_from_drawn, C_IN
from model import DuelingQNet, q_from
from buffer import Replay, NStepTracker

ok = lambda m: print("PASS  " + m, flush=True)

for N in (3, 5, 6):
    g = Geometry(N)
    b = Batch(g, 16, np.random.default_rng(N))
    rng = np.random.default_rng(N)
    while not b.done.all():
        a = np.array([rng.choice(np.flatnonzero(b.legal()[i])) if not b.done[i] else 0
                      for i in range(16)])
        b.step(a)
    assert (b.score.sum(1) == g.B).all(), "every box scored exactly once"
    assert (b.drawn == 1).all(), "every line drawn"
ok("random games terminate with every box scored exactly once (3x3, 5x5, 6x6)")

g = Geometry(5)
b = Batch(g, 8, np.random.default_rng(0))
rng = np.random.default_rng(0)
for _ in range(30):
    b.step(np.array([rng.choice(np.flatnonzero(b.legal()[i])) for i in range(8)]))
X1 = b.encode()
enc = Encoder(g, 8)
X2 = enc(b.drawn, sides_from_drawn(g, b.drawn), b.owner, b.score, b.mover)
assert np.array_equal(X1, X2) and np.isfinite(X1).all()
ok("self-play encoder and replay encoder produce identical features")

# the sum of signed per-move rewards must equal the true final margin
b = Batch(g, 32, np.random.default_rng(3))
rng = np.random.default_rng(3)
acc = np.zeros(32)
sign = np.ones(32)
start = b.mover.copy()
while not b.done.all():
    live = ~b.done
    a = np.array([rng.choice(np.flatnonzero(b.legal()[i])) if live[i] else 0 for i in range(32)])
    r, same = b.step(a)
    acc += np.where(live, sign * r, 0.0)
    sign = np.where(live & ~same, -sign, sign)
true = b.margin() / g.B
assert np.allclose(acc, np.where(start == 1, true, -true), atol=1e-6), (acc[:4], true[:4])
ok("signed rewards telescope exactly to the final margin (no shaping bias)")

net = DuelingQNet(16, 2)
lf = torch.as_tensor((g.line_cell[:, 0] * g.S + g.line_cell[:, 1]).astype(np.int64))
x = torch.randn(4, C_IN, g.S, g.S)
adv, v = net(x, lf)
legal = torch.zeros(4, g.L, dtype=torch.bool)
legal[:, :7] = True
q = q_from(adv, v, legal)
assert torch.isfinite(q[:, :7]).all() and (q[:, 7:] < -1e8).all()
ok("dueling head: legal Q finite, illegal moves masked out")

tr = NStepTracker(1, 3, 1.0)
st = lambda i: (np.zeros(g.L, np.uint8) + i, np.zeros(g.B, np.int8), np.zeros(2, np.int16), np.int8(1))
for i, (r, same) in enumerate([(0.0, False), (0.2, True), (0.0, False), (0.0, False)]):
    tr.push(0, st(i), i, r, same)
out = tr.pop_ready(0, st(9), True)
assert len(out) == 4 and out[0]["k"] == 3
# walk it by hand from step 0. move 0 does not capture -> turn passes to the
# opponent; move 1 captures 0.2 for the OPPONENT and keeps their turn; move 2
# does not capture -> the turn comes back.  So from step 0's point of view the
# accumulated reward is -0.2 and the state we bootstrap from is ours again.
assert abs(out[0]["r_acc"] + 0.2) < 1e-6, out[0]["r_acc"]
assert out[0]["sign"] == +1, out[0]["sign"]
assert out[0]["n_drawn"][0] == 3 and out[0]["done"] == 0, "bootstrap from the 3-steps-later state"
# the last transition has nothing after it, so it must bootstrap from the end
assert out[-1]["done"] == 1 and out[-1]["k"] == 1
ok("n-step returns carry the correct sign through the extra-turn rule")
print("ALL SELF-TESTS PASSED", flush=True)


In [ ]:
# ==== DOTS & BOXES - DQN training launcher (Kaggle, GPU T4 x2) ====
# Two DQN runs side by side, one per GPU, same code, different hyper-parameters.
# A: steady   - slow exploration decay, small steps, 3-step returns
# B: aggressive - fast decay, bigger steps, 5-step returns, more updates
# Both stream their logs here, checkpoint every 15 min, and stop before the budget.
import os, sys, time, json, glob, shutil, signal, threading, subprocess

TOTAL_HOURS = float(os.environ.get("DQN_HOURS", 8.5))
WORK = os.environ.get("DQN_WORK", "/kaggle/working")
BOARD = 5
T0 = time.time()
os.chdir(WORK)


def say(m):
    print(time.strftime("%H:%M:%S ") + "[launcher] " + m, flush=True)


try:
    import numba  # noqa
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numba"], check=True)
import torch, numba

ngpu = torch.cuda.device_count()
mem_gb = int(open("/proc/meminfo").readline().split()[1]) / 1e6
say(f"python {sys.version.split()[0]} | torch {torch.__version__} | numba {numba.__version__} | "
    f"cpus {os.cpu_count()} | RAM {mem_gb:.1f} GB | "
    f"GPUs {[torch.cuda.get_device_name(i) for i in range(ngpu)]}")
if ngpu == 0:
    say("WARNING: no GPU - turn on 'GPU T4 x2' in Settings > Accelerator.")

# --- sanity check the game core before spending 8 hours on it -----------------
chk = subprocess.run([sys.executable, "selftest.py"], capture_output=True, text=True)
print(chk.stdout[-4000:], chk.stderr[-3000:], flush=True)
if chk.returncode != 0:
    raise SystemExit("self-test failed - not starting training")

# --- resume from a previous notebook's output, if one is attached -------------
def find_resume(name):
    hits = sorted(glob.glob(f"/kaggle/input/**/runs/{name}/latest.pt", recursive=True))
    return os.path.dirname(hits[-1]) if hits else None

hours = TOTAL_HOURS - (time.time() - T0) / 3600 - 0.05
games = 384 if mem_gb >= 20 else 192

RUNS = [
    ("dqn_a", ["--lr", "3e-4", "--nstep", "3", "--eps-decay-steps", "4000000",
               "--target-sync", "2000", "--updates", "8", "--collect-steps", "8"]),
    ("dqn_b", ["--lr", "1e-3", "--nstep", "5", "--eps-decay-steps", "1500000",
               "--target-sync", "1000", "--updates", "12", "--collect-steps", "8"]),
]
say(f"starting {len(RUNS)} DQN runs, {hours:.2f} h each, {games} parallel games, board {BOARD}x{BOARD}")

procs = []
for i, (name, extra) in enumerate(RUNS):
    env = dict(os.environ, PYTHONUNBUFFERED="1", NUMBA_CACHE_DIR=f"/tmp/numba_{name}")
    if ngpu:
        env["CUDA_VISIBLE_DEVICES"] = str(i % ngpu)
    cmd = [sys.executable, "-u", "train_dqn.py", "--board", str(BOARD), "--out", f"runs/{name}",
           "--hours", f"{hours:.3f}", "--games", str(games), "--channels", "64", "--blocks", "6",
           "--batch", "512", "--buffer", "600000", "--min-buffer", "50000",
           "--eval-minutes", "20", "--eval-games", "200", "--save-minutes", "15",
           "--seed", str(1234 + i)] + extra
    res = find_resume(name)
    if res:
        cmd += ["--resume-from", res]
        say(f"{name}: resuming from {res}")
    say(f"{name}: " + " ".join(cmd[2:]))
    procs.append((name, subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE,
                                         stderr=subprocess.STDOUT, text=True, bufsize=1)))


def relay(name, p):
    for line in p.stdout:
        print(f"[{name}] {line}", end="", flush=True)


threads = [threading.Thread(target=relay, args=x, daemon=True) for x in procs]
for t in threads:
    t.start()

last_beat = time.time()
sent = False
while any(p.poll() is None for _, p in procs):
    el = (time.time() - T0) / 3600
    if el > TOTAL_HOURS + 0.05 and not sent:
        say("budget reached - asking both runs to save and stop")
        for _, p in procs:
            if p.poll() is None:
                p.send_signal(signal.SIGTERM)
        sent = True
    if el > TOTAL_HOURS + 0.2:
        say("hard stop")
        for _, p in procs:
            if p.poll() is None:
                p.kill()
    if time.time() - last_beat > 900:
        last_beat = time.time()
        try:
            smi = subprocess.run(["nvidia-smi", "--query-gpu=index,utilization.gpu,memory.used",
                                  "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
        except Exception:
            smi = "n/a"
        say(f"heartbeat {el:.2f} h | GPU: {smi.replace(chr(10), ' ; ')}")
    time.sleep(20)

for t in threads:
    t.join(timeout=10)
say("return codes: " + str({n: p.returncode for n, p in procs}))

os.makedirs("models", exist_ok=True)
for name, _ in RUNS:
    for f in ("best.pt", "latest.pt"):
        src = f"runs/{name}/{f}"
        if os.path.exists(src):
            shutil.copy2(src, f"models/{name}_{f}")
    st = f"runs/{name}/status.json"
    if os.path.exists(st):
        say(f"{name} final status: " + open(st).read())
say(f"all done in {(time.time() - T0) / 3600:.2f} h. Download models/dqn_a_best.pt and models/dqn_b_best.pt")
